<a href="https://colab.research.google.com/github/Meghnashankr23/3DVSS_June2026/blob/main/Copy_of_LoRa_Finetuning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
! unzip /content/output.zip

Archive:  /content/output.zip
   creating: output/
  inflating: __MACOSX/._output       
   creating: output/darkbrown/
  inflating: __MACOSX/output/._darkbrown  
  inflating: output/.DS_Store        
  inflating: __MACOSX/output/._.DS_Store  
   creating: output/lightbrown/
  inflating: __MACOSX/output/._lightbrown  
   creating: output/croatian/
  inflating: __MACOSX/output/._croatian  
   creating: output/.ipynb_checkpoints/
  inflating: __MACOSX/output/._.ipynb_checkpoints  
   creating: output/white/
  inflating: __MACOSX/output/._white  
   creating: output/black/
  inflating: __MACOSX/output/._black  
  inflating: output/darkbrown/rough.png  
  inflating: __MACOSX/output/darkbrown/._rough.png  
  inflating: output/darkbrown/young.png  
  inflating: __MACOSX/output/darkbrown/._young.png  
  inflating: output/darkbrown/ hair.png  
  inflating: __MACOSX/output/darkbrown/._ hair.png  
  inflating: output/darkbrown/old skin with glittery nails.png  
  inflating: __MACOSX/output/darkb

In [ ]:
!unzip /content/output_filename.zip

Archive:  /content/output_filename.zip
   creating: content/lora_back_v1/
   creating: content/lora_back_v1/checkpoint-14000/
  inflating: content/lora_back_v1/checkpoint-14000/pytorch_lora_weights.safetensors  
  inflating: content/lora_back_v1/checkpoint-14000/random_states_0.pkl  
  inflating: content/lora_back_v1/checkpoint-14000/optimizer.bin  
  inflating: content/lora_back_v1/checkpoint-14000/scheduler.bin  
  inflating: content/lora_back_v1/checkpoint-14000/scaler.pt  
   creating: content/lora_back_v1/checkpoint-8000/
  inflating: content/lora_back_v1/checkpoint-8000/pytorch_lora_weights.safetensors  
  inflating: content/lora_back_v1/checkpoint-8000/random_states_0.pkl  
  inflating: content/lora_back_v1/checkpoint-8000/optimizer.bin  
  inflating: content/lora_back_v1/checkpoint-8000/scheduler.bin  
  inflating: content/lora_back_v1/checkpoint-8000/scaler.pt  
   creating: content/lora_back_v1/checkpoint-13000/
  inflating: content/lora_back_v1/checkpoint-13000/pytorch_lora_

In [ ]:
import os
import cv2
import numpy as np
import pandas as pd

# ================= CONFIGURATION =================
train_dir = "train"              # Your source folder
mask_path = "semantic_mask.png"  # Your mask
target_size = (768,768)

# OUTPUT CONFIGURATION
# We save metadata.csv INSIDE the folder so the training script finds it automatically
dir_palm = "dataset_palm"
dir_back = "dataset_back"
csv_palm = os.path.join(dir_palm, "metadata.csv")
csv_back = os.path.join(dir_back, "metadata.csv")
# =================================================

# 1. Load and Prepare Mask
mask_img = cv2.imread(mask_path)
mask_img = cv2.resize(mask_img, target_size, interpolation=cv2.INTER_NEAREST)

# OpenCV loads images as BGR (Blue, Green, Red)
# Channel 0 = Blue
# Channel 1 = Green
# Channel 2 = Red

# CORRECTED LOGIC:
# Blue Channel > 100 = Back of Hand
mask_back = (mask_img[:, :, 0] > 100).astype(np.uint8) * 255
# Red Channel > 100 = Palm
mask_palm = (mask_img[:, :, 2] > 100).astype(np.uint8) * 255

os.makedirs(dir_palm, exist_ok=True)
os.makedirs(dir_back, exist_ok=True)

data_palm = []
data_back = []

print("Splitting dataset into Palm and Back...")

for root, dirs, files in os.walk(train_dir):
    for filename in files:
        if filename.lower().endswith(('.png', '.jpg', '.jpeg')):

            # Paths & Info
            file_path = os.path.join(root, filename)
            folder_name = os.path.basename(root)      # Skin Type (e.g., "white")
            file_base = os.path.splitext(filename)[0] # Variation (e.g., "young with glittery nails")

            if root == train_dir: continue

            # Clean text for the PROMPT (Remove underscores, keep spaces)
            skin = folder_name.replace("_", " ")
            var = file_base.replace("_", " ")

            # Clean text for the FILENAME (Optional: replace spaces with underscores to be safe)
            # If you prefer keeping spaces in filenames, you can remove the .replace below
            safe_var = file_base.replace(" ", "_")
            safe_skin = folder_name.replace(" ", "_")

            try:
                # Load & Resize
                img = cv2.imread(file_path)
                img = cv2.resize(img, target_size, interpolation=cv2.INTER_LANCZOS4)

                # --- 1. CREATE PALM DATA ---
                img_palm = cv2.bitwise_and(img, img, mask=mask_palm)

                # Naming convention: white_young_with_glittery_nails_palm.png
                save_name_palm = f"{safe_skin}_{safe_var}_palm.png"
                save_path_palm = os.path.join(dir_palm, save_name_palm)
                cv2.imwrite(save_path_palm, img_palm)

                # Metadata: We only store the FILENAME (not the full path)
                data_palm.append({
                    "file_name": save_name_palm,
                    "text": f"photo of a human palm, {skin}, {var}, smooth skin, lines, high detail, 8k, uv layout"
                })

                # --- 2. CREATE BACK DATA ---
                img_back = cv2.bitwise_and(img, img, mask=mask_back)

                save_name_back = f"{safe_skin}_{safe_var}_back.png"
                save_path_back = os.path.join(dir_back, save_name_back)
                cv2.imwrite(save_path_back, img_back)

                # Metadata
                data_back.append({
                    "file_name": save_name_back,
                    "text": f"photo of the back of a human hand, {skin}, {var}, pores, wrinkles, knuckles, veins, 8k, uv layout"
                })

            except Exception as e:
                print(f"Error on {filename}: {e}")

# Save CSVs
pd.DataFrame(data_palm).to_csv(csv_palm, index=False)
pd.DataFrame(data_back).to_csv(csv_back, index=False)

print(f"✅ Done.")
print(f"   Palm dataset saved to: {dir_palm} (with metadata.csv inside)")
print(f"   Back dataset saved to: {dir_back} (with metadata.csv inside)")

Splitting dataset into Palm and Back...
✅ Done.
   Palm dataset saved to: dataset_palm (with metadata.csv inside)
   Back dataset saved to: dataset_back (with metadata.csv inside)


In [ ]:
!pip install diffusers transformers accelerate bitsandbytes

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 20.4 MB/s eta 0:00:00


In [ ]:
!pip install git+https://github.com/huggingface/diffusers.git

  Cloning https://github.com/huggingface/diffusers.git to /tmp/pip-req-build-xua7bf2o
  Running command git clone --filter=blob:none --quiet https://github.com/huggingface/diffusers.git /tmp/pip-req-build-xua7bf2o
  Resolved https://github.com/huggingface/diffusers.git to commit 9a72cd3ee9eaefbf5cac47640ba1c3acf082634d
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Created wheel for diffusers: filename=diffusers-0.39.0.dev0-py3-none-any.whl size=5593062 sha256=cd6c7cefa1280cb87c09047b469abce78059503625e1df9e3efd0ac5c38dba23
  Stored in directory: /tmp/pip-ephem-wheel-cache-l23n2d6c/wheels/23/0f/7d/f97813d265ed0e599a78d83afd4e1925740896ca79b46cccfd
Successfully built diffusers
  Attempting uninstall: diffusers
    Found existing installation: diffusers 0.38.0
    Uninstalling diffusers-0.38.0:
      Successfully uninstalled diffusers-0.38.0


In [ ]:
!wget https://raw.githubusercontent.com/huggingface/diffusers/v0.36.0/examples/text_to_image/train_text_to_image_lora.py

--2026-07-02 14:45:34--  https://raw.githubusercontent.com/huggingface/diffusers/v0.36.0/examples/text_to_image/train_text_to_image_lora.py
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 43229 (42K) [text/plain]
Saving to: ‘train_text_to_image_lora.py’

train_text_to_image 100%[===================>]  42.22K  --.-KB/s    in 0s      

2026-07-02 14:45:34 (145 MB/s) - ‘train_text_to_image_lora.py’ saved [43229/43229]



In [ ]:
import os
import shutil

# Define your paths
folders = [
    ("dataset_palm", "metadata_palm.csv"),
    ("dataset_back", "metadata_back.csv")
]

for folder_name, csv_name in folders:
    # Check if the folder and CSV exist
    if os.path.exists(folder_name) and os.path.exists(csv_name):
        destination = os.path.join(folder_name, "metadata.csv")

        # Move and Rename
        shutil.move(csv_name, destination)
        print(f"✅ Fixed: Moved '{csv_name}' to '{destination}'")
    else:
        print(f"⚠️ Warning: Could not find '{folder_name}' or '{csv_name}'. Check if they exist.")

print("\nReady to train!")

⚠️ Warning: Could not find 'dataset_palm' or 'metadata_palm.csv'. Check if they exist.
⚠️ Warning: Could not find 'dataset_back' or 'metadata_back.csv'. Check if they exist.

Ready to train!


In [ ]:
import pandas as pd
import os

# The folders we need to fix
folders = ["dataset_palm", "dataset_back"]

for folder in folders:
    csv_path = os.path.join(folder, "metadata.csv")

    if os.path.exists(csv_path):
        print(f"Fixing paths in {csv_path}...")

        # Read the CSV
        df = pd.read_csv(csv_path)

        # Function to strip the folder name from the path
        def clean_path(path):
            # Normalize slashes just in case
            path = path.replace("\\", "/")
            # If the path starts with "dataset_palm/", remove it
            if path.startswith(f"{folder}/"):
                return path.replace(f"{folder}/", "")
            # Fallback: Just return the filename itself
            return os.path.basename(path)

        # Apply the fix
        df['file_name'] = df['file_name'].apply(clean_path)

        # Save it back (overwrite)
        df.to_csv(csv_path, index=False)

        print(f"✅ Success! First entry is now: {df.iloc[0]['file_name']}")
    else:
        print(f"❌ Error: Could not find {csv_path}")

print("\nPaths are clean. You can run training now.")

Fixing paths in dataset_palm/metadata.csv...
✅ Success! First entry is now: white_young_skin_with_engagement_ring_palm.png
Fixing paths in dataset_back/metadata.csv...
✅ Success! First entry is now: white_young_skin_with_engagement_ring_back.png

Paths are clean. You can run training now.


In [ ]:
!accelerate launch train_text_to_image_lora.py \
  --pretrained_model_name_or_path="SG161222/Realistic_Vision_V5.1_noVAE" \
  --train_data_dir="dataset_palm" \
  --caption_column="text" \
  --resolution=768 \
  --random_flip \
  --train_batch_size=1 \
  --num_train_epochs=350 \
  --learning_rate=1e-04 \
  --output_dir="lora_palm_v1" \
  --mixed_precision="fp16" \
  --checkpointing_steps=1000 \
  --resume_from_checkpoint="latest"

The following values were not passed to `accelerate launch` and had defaults used instead:
	`--num_processes` was set to a value of `1`
	`--num_machines` was set to a value of `1`
	`--mixed_precision` was set to a value of `'no'`
	`--dynamo_backend` was set to a value of `'no'`
To avoid this warning pass in values for each of the problematic parameters or run `accelerate config`.
Flax classes are deprecated and will be removed in Diffusers v1.0.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.
Flax classes are deprecated and will be removed in Diffusers v1.0.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.
INFO:__main__:Distributed environment: DistributedType.NO
Num processes: 1
Process index: 0
Local process index: 0
Device: cuda

Mixed precision type: fp16

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_validators.py:202: UserWarning: The `local_dir_use_symlinks` argument is deprecated and ignored in 

In [ ]:
!accelerate launch train_text_to_image_lora.py \
  --pretrained_model_name_or_path="SG161222/Realistic_Vision_V5.1_noVAE" \
  --train_data_dir="dataset_back" \
  --caption_column="text" \
  --resolution=768 \
  --random_flip \
  --train_batch_size=1 \
  --num_train_epochs=350 \
  --learning_rate=1e-04 \
  --output_dir="lora_back_v1" \
  --mixed_precision="fp16" \
 --resume_from_checkpoint="latest" \
  --checkpointing_steps=1000

The following values were not passed to `accelerate launch` and had defaults used instead:
	`--num_processes` was set to a value of `1`
	`--num_machines` was set to a value of `1`
	`--mixed_precision` was set to a value of `'no'`
	`--dynamo_backend` was set to a value of `'no'`
To avoid this warning pass in values for each of the problematic parameters or run `accelerate config`.
Flax classes are deprecated and will be removed in Diffusers v1.0.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.
Flax classes are deprecated and will be removed in Diffusers v1.0.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.
INFO:__main__:Distributed environment: DistributedType.NO
Num processes: 1
Process index: 0
Local process index: 0
Device: cuda

Mixed precision type: fp16

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_validators.py:202: UserWarning: The `local_dir_use_symlinks` argument is deprecated and ignored in 

In [ ]:
import cv2
import torch
import numpy as np
from PIL import Image
from diffusers import StableDiffusionControlNetPipeline, ControlNetModel, UniPCMultistepScheduler

# ================= CONFIGURATION =================
base_model = "SG161222/Realistic_Vision_V5.1_noVAE"
path_lora_back = "/content/back"
path_lora_palm = "/content/palm"

# INPUT MAPS (Ensure these exist!)
path_depth  = "/content/uv_depth.png"   # Grayscale Depth Map
path_normal = "/content/controlnet_uv_normal_ring.png"  # Purple Normal Map
mask_path   = "semantic_mask.png"

# RESOLUTION SETTING (Locked to 768)
target_size = (768, 768)
# =================================================

# 1. Load Multi-ControlNet Pipeline
print("Loading models...")
# Load Depth and Normal ControlNets
cn_depth  = ControlNetModel.from_pretrained("lllyasviel/control_v11f1p_sd15_depth", torch_dtype=torch.float16)
cn_normal = ControlNetModel.from_pretrained("lllyasviel/control_v11p_sd15_normalbae", torch_dtype=torch.float16)

# Pass them as a list: [Depth, Normal]
pipe = StableDiffusionControlNetPipeline.from_pretrained(
    base_model,
    controlnet=[cn_depth, cn_normal],
    torch_dtype=torch.float16
).to("cuda")
pipe.scheduler = UniPCMultistepScheduler.from_config(pipe.scheduler.config)

# 2. Prepare Inputs
print(f"Preparing inputs at {target_size}...")

def load_image(path, mode="RGB"):
    # Resize to target_size (768x768) to match generation output
    return Image.open(path).convert(mode).resize(target_size, Image.Resampling.LANCZOS)

# Load Control Images
img_depth  = load_image(path_depth, "RGB")
img_normal = load_image(path_normal, "RGB")
control_images = [img_depth, img_normal] # List matching the model order [Depth, Normal]

# Load Mask (Nearest Neighbor to keep sharp edges)
mask_raw = Image.open(mask_path).convert("RGB").resize(target_size, Image.Resampling.NEAREST)
mask_arr = np.array(mask_raw)

# Create binary masks (Blue=Back, Red=Palm)
# Note: In PIL RGB, Red is index 0, Blue is index 2.
mask_palm_final = Image.fromarray((mask_arr[:,:,0] > 100).astype(np.uint8) * 255).convert("L")
mask_back_final = Image.fromarray((mask_arr[:,:,2] > 100).astype(np.uint8) * 255).convert("L")

def generate_perfect_hand(skin, variation):
    print(f"--- Generative Process: {skin} {variation} ---")

    # Common Inference Settings
    # We set scales: Depth=1.0 (Strong structure), Normal=0.8 (Texture details without noise)
    gen_kwargs = {
        "image": control_images,
        "height": target_size[0], # <--- CRITICAL FIX: 768
        "width": target_size[1],  # <--- CRITICAL FIX: 768
        "num_inference_steps":30,
        "guidance_scale":3,
        "controlnet_conditioning_scale": [0.8, 0.5]
    }

    # --- STEP A: GENERATE PALM ---
    print("1. Generating Palm...")
    pipe.unload_lora_weights()
    pipe.load_lora_weights(path_lora_palm)

    prompt_palm = f"photo of a human palm, {skin}, {variation}, smooth skin, lines, hyper-realistic, 8k, uv layout"

    image_palm = pipe(prompt_palm, **gen_kwargs).images[0]

    # --- STEP B: GENERATE BACK ---
    print("2. Generating Back...")
    pipe.unload_lora_weights()
    pipe.load_lora_weights(path_lora_back)

    prompt_back = f"photo of the back of a human hand, {skin}, {variation}, pores, wrinkles, knuckles, veins, 8k, uv layout"

    image_back = pipe(prompt_back, **gen_kwargs).images[0]

    # --- STEP C: STITCH TOGETHER ---
    print("3. Stitching...")

    # Start with black background (Now 768x768)
    final_comp = Image.new("RGB", target_size, (0, 0, 0))

    # Paste Palm using Palm Mask
    # NOW: Image is 768, Mask is 768. It will work.
    final_comp.paste(image_palm, (0,0), mask_palm_final)

    # Paste Back using Back Mask
    final_comp.paste(image_back, (0,0), mask_back_final)

    # Save
    filename = f"final_{skin.replace(' ','_')}_{variation.replace(' ','_')}.png"
    final_comp.save(filename)
    print(f"✅ Saved to {filename}")

# RUN
generate_perfect_hand("black", "old skin with glittery nails")
generate_perfect_hand("brown", "young skin with engagement ring")

Flax classes are deprecated and will be removed in Diffusers v0.40.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.
Flax classes are deprecated and will be removed in Diffusers v0.40.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_validators.py:205: UserWarning: The `local_dir_use_symlinks` argument is deprecated and ignored in `hf_hub_download`. Downloading to a local directory does not use symlinks anymore.
  warnings.warn(


Loading models...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/945 [00:00<?, ?B/s]

diffusion_pytorch_model.safetensors:   0%|          | 0.00/1.45G [00:00<?, ?B/s]

...

Fetching 12 files:   0%|          | 0/12 [00:00<?, ?it/s]

Loading pipeline components...:   0%|          | 0/5 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/196 [00:00<?, ?it/s]

Preparing inputs at (768, 768)...


FileNotFoundError: [Errno 2] No such file or directory: '/content/uv_depth.png'

In [ ]:
import zipfile
import os

zip_file_path = '/content/lora_back_v1.zip'
output_directory = '/content/' # Extract to the content directory

# Create the output directory if it doesn't exist
os.makedirs(output_directory, exist_ok=True)

with zipfile.ZipFile(zip_file_path, 'r') as zip_ref:
    zip_ref.extractall(output_directory)

print(f"Successfully extracted {zip_file_path} to {output_directory}")

zip_file_path = '/content/lora_palm_v1.zip'
output_directory = '/content/' # Extract to the content directory

# Create the output directory if it doesn't exist
os.makedirs(output_directory, exist_ok=True)

with zipfile.ZipFile(zip_file_path, 'r') as zip_ref:
    zip_ref.extractall(output_directory)

print(f"Successfully extracted {zip_file_path} to {output_directory}")

Successfully extracted /content/lora_back_v1.zip to /content/
Successfully extracted /content/lora_palm_v1.zip to /content/


In [ ]:
import cv2
import torch
import numpy as np
from PIL import Image
from diffusers import StableDiffusionControlNetPipeline, ControlNetModel, UniPCMultistepScheduler

# ================= CONFIGURATION =================
base_model = "SG161222/Realistic_Vision_V5.1_noVAE"
path_lora_back = "/content/lora_back_v1/pytorch_lora_weights.safetensors"
path_lora_palm = "/content/lora_palm_v1/pytorch_lora_weights.safetensors"

# INPUT MAPS (You need to upload these)
path_depth  = "/content/uv_depth.png"
path_normal = "/content/control_uv_normal.png"
mask_path   = "/content/semantic_mask.png"

# RESOLUTION SETTING (Locked to 768)
target_size = (768, 768)
# =================================================

# 1. Load Multi-ControlNet Pipeline
print("Loading models...")
# Load Depth and Normal ControlNets
cn_depth  = ControlNetModel.from_pretrained("lllyasviel/control_v11f1p_sd15_depth", torch_dtype=torch.float16)
cn_normal = ControlNetModel.from_pretrained("lllyasviel/control_v11p_sd15_normalbae", torch_dtype=torch.float16)

# Pass them as a list: [Depth, Normal]
pipe = StableDiffusionControlNetPipeline.from_pretrained(
    base_model,
    controlnet=[cn_depth, cn_normal],
    torch_dtype=torch.float16
).to("cuda")
pipe.scheduler = UniPCMultistepScheduler.from_config(pipe.scheduler.config)

# 2. Prepare Inputs
print(f"Preparing inputs at {target_size}...")

def load_image(path, mode="RGB"):
    return Image.open(path).convert(mode).resize(target_size, Image.Resampling.LANCZOS)

# Load Control Images
img_depth  = load_image(path_depth, "RGB")
img_normal = load_image(path_normal, "RGB")
control_images = [img_depth, img_normal] # List matching the model order

# Load Mask (Nearest Neighbor to keep sharp edges)
mask_raw = Image.open(mask_path).convert("RGB").resize(target_size, Image.Resampling.NEAREST)
mask_arr = np.array(mask_raw)

# Create binary masks (Blue=Back, Red=Palm)
# Note: In PIL RGB, Red is index 0, Blue is index 2.
# Adjust indices [0] or [2] below if your mask colors are swapped.
mask_palm_final = Image.fromarray((mask_arr[:,:,0] > 100).astype(np.uint8) * 255).convert("L")
mask_back_final = Image.fromarray((mask_arr[:,:,2] > 100).astype(np.uint8) * 255).convert("L")

def generate_perfect_hand(skin, variation):
    print(f"--- Generative Process: {skin} {variation} ---")

    # Common Inference Settings
    # We set scales: Depth=1.0 (Strong structure), Normal=0.8 (Texture details without noise)
    gen_kwargs = {
        "image": control_images,
        "height": target_size[0],
        "width": target_size[1],
        "num_inference_steps":25,
        "guidance_scale":3.5,
        "controlnet_conditioning_scale": [0.6, 0.8]
    }

    # --- STEP A: GENERATE PALM ---
    print("1. Generating Palm...")
    pipe.unload_lora_weights()
    pipe.load_lora_weights(path_lora_palm)

    prompt_palm = f"photo of a human palm, {skin}, {variation}, smooth skin, lines, hyper-realistic, 8k, uv layout"

    image_palm = pipe(prompt_palm, **gen_kwargs).images[0]

    # --- STEP B: GENERATE BACK ---
    print("2. Generating Back...")
    pipe.unload_lora_weights()
    pipe.load_lora_weights(path_lora_back)

    prompt_back = f"photo of the back of a human hand, {skin}, {variation}, pores, wrinkles, knuckles, veins, 8k, uv layout"

    image_back = pipe(prompt_back, **gen_kwargs).images[0]

    # --- STEP C: STITCH TOGETHER ---
    print("3. Stitching...")

    # Start with black background
    final_comp = Image.new("RGB", target_size, (0, 0, 0))

    # Paste Palm using Palm Mask
    final_comp.paste(image_palm, (0,0), mask_palm_final)

    # Paste Back using Back Mask
    final_comp.paste(image_back, (0,0), mask_back_final)

    # Convert back image to numpy
    back_np = np.array(image_back)

    # Compute mean color of BACK image
    mean_color = tuple(np.mean(back_np.reshape(-1, 3), axis=0).astype(np.uint8))

    print(f"Background mean color: {mean_color}")

    # Create background using mean back color
    final_comp = Image.new("RGB", target_size, mean_color)

    # Paste Palm using Palm Mask
    final_comp.paste(image_palm, (0,0), mask_palm_final)

    # Paste Back using Back Mask
    final_comp.paste(image_back, (0,0), mask_back_final)

    # Save
    filename = f"final_{skin.replace(' ','_')}_{variation.replace(' ','_')}.png"
    final_comp.save(filename)

    print(f"✅ Saved to {filename}")


# RUN
generate_perfect_hand("white", "young skin")
generate_perfect_hand("darkbrown", "old skin with glittery nails ring")

Loading models...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_validators.py:205: UserWarning: The `local_dir_use_symlinks` argument is deprecated and ignored in `hf_hub_download`. Downloading to a local directory does not use symlinks anymore.
  warnings.warn(


Loading pipeline components...:   0%|          | 0/5 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/196 [00:00<?, ?it/s]

Preparing inputs at (768, 768)...
--- Generative Process: white young skin ---
1. Generating Palm...


...

  0%|          | 0/25 [00:00<?, ?it/s]

2. Generating Back...


No LoRA keys associated to CLIPTextModel found with the prefix='text_encoder'. This is safe to ignore if LoRA state dict didn't originally have any CLIPTextModel related params. You can also try specifying `prefix=None` to resolve the warning. Otherwise, open an issue if you think it's unexpected: https://github.com/huggingface/diffusers/issues/new


  0%|          | 0/25 [00:00<?, ?it/s]

3. Stitching...
Background mean color: (np.uint8(45), np.uint8(23), np.uint8(20))
✅ Saved to final_darkbrown_old_skin_with_glittery_nails_ring.png


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
!zip -r /content/LoRa_Textures.zip /content/lora_back_v1 /content/lora_palm_v1

  adding: content/lora_back_v1/ (stored 0%)
  adding: content/lora_back_v1/pytorch_lora_weights.safetensors (deflated 6%)
  adding: content/lora_back_v1/logs/ (stored 0%)
  adding: content/lora_back_v1/logs/text2image-fine-tune/ (stored 0%)
  adding: content/lora_back_v1/logs/text2image-fine-tune/1770200510.031525/ (stored 0%)
  adding: content/lora_back_v1/logs/text2image-fine-tune/1770200510.031525/events.out.tfevents.1770200510.92ef51d26f57.20760.1 (deflated 52%)
  adding: content/lora_back_v1/logs/text2image-fine-tune/events.out.tfevents.1770200510.92ef51d26f57.20760.0 (deflated 66%)
  adding: content/lora_back_v1/logs/text2image-fine-tune/1770197736.569725/ (stored 0%)
  adding: content/lora_back_v1/logs/text2image-fine-tune/1770197736.569725/hparams.yml (deflated 49%)
  adding: content/lora_back_v1/logs/text2image-fine-tune/1770200510.0329661/ (stored 0%)
  adding: content/lora_back_v1/logs/text2image-fine-tune/1770200510.0329661/hparams.yml (deflated 49%)
  adding: content/lora_

In [ ]:
!pip install --upgrade torchao>=0.16.0

In [ ]:

!pip install "pyvista[jupyter]"
import pyvista as pv
pv.set_jupyter_backend('html')
pv.OFF_SCREEN = False

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 221.2/221.2 kB 12.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 251.7/251.7 kB 26.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.4/44.4 kB 4.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 831.8/831.8 kB 51.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.1/5.1 MB 129.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 146.0/146.0 MB 1.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.1/45.1 kB 440.7 kB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.6/2.6 MB 9.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 272.9/272.9 kB 29.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 99.6 MB/s eta 0:00:00


In [ ]:
texture = pv.read_texture('/content/final_darkbrown_old_skin_with_glittery_nails_ring.png')
plotter = pv.Plotter()
mesh = pv.read("/content/MANO_UV_right.obj")
plotter.add_mesh(mesh, texture=texture)
plotter.show()

EmbeddableWidget(value='<iframe srcdoc="<!doctype html>\n<html lang=&quot;en&quot;>\n  <head>\n    <meta chars…

### Convert Notebook to Python Script

To convert this Colab notebook (`.ipynb` file) into a Python script (`.py` file), you can use the `jupyter nbconvert` command.

First, you need to find the exact name of your notebook file. You can usually find this at the top of your browser tab or by listing the files in your current directory using `!ls`.

Then, replace `YourNotebookName.ipynb` in the command below with the actual name of your notebook.

In [ ]:
!pip list


Package                                  Version
---------------------------------------- ------------------
absl-py                                  1.4.0
accelerate                               1.13.0
access                                   1.1.10.post3
affine                                   2.4.0
aiofiles                                 24.1.0
aiohappyeyeballs                         2.6.2
aiohttp                                  3.14.1
aiosignal                                1.4.0
aiosqlite                                0.22.1
alabaster                                1.0.0
albucore                                 0.0.24
albumentations                           2.0.8
ale-py                                   0.12.0
alembic                                  1.18.4
altair                                   5.5.0
annotated-doc                            0.0.4
annotated-types                          0.7.0
antlr4-python3-runtime                   4.9.3
anyio                          